In [13]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import json
import sqlite3
from faker import Faker

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


In [14]:
# ============================================================================
# 1. EXTRACT INDIGO FLIGHTS FROM airline_routes.json
# ============================================================================

with open('airline_routes.json', 'r') as f:
    routes_data = json.load(f)

# Extract Indigo routes (look for all routes from the JSON structure)
indigo_routes = []

for airport_code, airport_data in routes_data.items():
    if isinstance(airport_data, dict) and 'routes' in airport_data:
        for route in airport_data['routes']:
            if isinstance(route, dict) and 'carriers' in route:
                for carrier in route['carriers']:
                    if carrier.get('iata') == '6E':  # Indigo code
                        indigo_routes.append({
                            'origin': airport_code,
                            'destination': route.get('iata'),
                            'distance_km': route.get('km', 0),
                            'duration_mins': route.get('min', 0),
                            'airline_code': '6E',
                            'airline_name': 'IndiGo'
                        })

print(f"Total Indigo routes extracted: {len(indigo_routes)}")
print(f"\nSample routes:")
for route in indigo_routes[:5]:
    print(f"  {route['origin']} → {route['destination']}: {route['distance_km']}km, {route['duration_mins']}min")

Total Indigo routes extracted: 1127

Sample routes:
  AGR → LKO: 293km, 75min
  AGR → BLR: 1545km, 140min
  AGR → BOM: 1034km, 115min
  AGR → HYD: 1100km, 120min
  AGX → COK: 466km, 85min


In [15]:
# ============================================================================
# 2. CREATE SQLITE DATABASE & SCHEMA
# ============================================================================

# Connect to database (creates if doesn't exist)
conn = sqlite3.connect('indigo_airline.db')
cursor = conn.cursor()

# Drop existing tables if they exist (for fresh start)
cursor.executescript('''
    DROP TABLE IF EXISTS FlightDelays;
    DROP TABLE IF EXISTS FlightInstances;
    DROP TABLE IF EXISTS PassengerBaggage;
    DROP TABLE IF EXISTS SpecialBaggage;
    DROP TABLE IF EXISTS Refunds;
    DROP TABLE IF EXISTS Payments;
    DROP TABLE IF EXISTS ItineraryLegs;
    DROP TABLE IF EXISTS Itineraries;
    DROP TABLE IF EXISTS Passengers;
    DROP TABLE IF EXISTS PNRs;
    DROP TABLE IF EXISTS Bookings;
    DROP TABLE IF EXISTS DaysOfOperation;
    DROP TABLE IF EXISTS FlightSchedule;
    DROP TABLE IF EXISTS Customers;
    DROP TABLE IF EXISTS ConnectionRules;
    DROP TABLE IF EXISTS AuditLog;
''')

# Create all tables
cursor.executescript('''
    -- Customers Table
    CREATE TABLE Customers (
        customer_id VARCHAR(20) PRIMARY KEY,
        first_name VARCHAR(100),
        last_name VARCHAR(100),
        email VARCHAR(255) UNIQUE,
        phone_number VARCHAR(20),
        date_of_birth DATE,
        gender VARCHAR(10),
        country_code CHAR(2),
        city VARCHAR(100),
        loyalty_program_id VARCHAR(20),
        customer_segment VARCHAR(50),
        created_at TIMESTAMP,
        updated_at TIMESTAMP,
        is_synthetic BOOLEAN DEFAULT 1
    );

    -- Flight Schedule (Master Data)
    CREATE TABLE FlightSchedule (
        flight_id VARCHAR(10) PRIMARY KEY,
        origin_airport_code CHAR(3),
        destination_airport_code CHAR(3),
        departure_time TIME,
        arrival_time TIME,
        flight_duration_minutes INTEGER,
        aircraft_type VARCHAR(20),
        seat_capacity INTEGER,
        status VARCHAR(50),
        created_at TIMESTAMP
    );

    -- Days of Operation
    CREATE TABLE DaysOfOperation (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        flight_id VARCHAR(10),
        day_of_week INTEGER,
        effective_from DATE,
        effective_to DATE,
        FOREIGN KEY (flight_id) REFERENCES FlightSchedule(flight_id)
    );

    -- PNRs
    CREATE TABLE PNRs (
        pnr_code VARCHAR(6) PRIMARY KEY,
        customer_id VARCHAR(20),
        pnr_status VARCHAR(50),
        issue_date DATETIME,
        valid_until DATE,
        remarks VARCHAR(500),
        created_at TIMESTAMP,
        updated_at TIMESTAMP,
        FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
    );

    -- Bookings
    CREATE TABLE Bookings (
        booking_id VARCHAR(20) PRIMARY KEY,
        pnr_code VARCHAR(6),
        customer_id VARCHAR(20),
        flight_id VARCHAR(10),
        flight_date DATE,
        total_passengers INTEGER,
        booking_status VARCHAR(50),
        booking_type VARCHAR(50),
        total_fare DECIMAL(10, 2),
        tax_charges DECIMAL(10, 2),
        discount DECIMAL(10, 2),
        final_amount DECIMAL(10, 2),
        created_at TIMESTAMP,
        updated_at TIMESTAMP,
        FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
        FOREIGN KEY (pnr_code) REFERENCES PNRs(pnr_code),
        FOREIGN KEY (flight_id) REFERENCES FlightSchedule(flight_id)
    );

    -- Passengers
    CREATE TABLE Passengers (
        passenger_id VARCHAR(20) PRIMARY KEY,
        booking_id VARCHAR(20),
        first_name VARCHAR(100),
        last_name VARCHAR(100),
        date_of_birth DATE,
        gender VARCHAR(10),
        passport_number VARCHAR(20),
        nationality CHAR(2),
        passenger_type VARCHAR(50),
        created_at TIMESTAMP,
        FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id)
    );

    -- Itineraries
    CREATE TABLE Itineraries (
        itinerary_id VARCHAR(20) PRIMARY KEY,
        booking_id VARCHAR(20),
        total_legs INTEGER,
        journey_type VARCHAR(50),
        total_duration_minutes INTEGER,
        created_at TIMESTAMP,
        FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id)
    );

    -- Itinerary Legs
    CREATE TABLE ItineraryLegs (
        leg_id VARCHAR(20) PRIMARY KEY,
        itinerary_id VARCHAR(20),
        leg_number INTEGER,
        flight_id VARCHAR(10),
        flight_date DATE,
        origin_airport CHAR(3),
        destination_airport CHAR(3),
        departure_time TIME,
        arrival_time TIME,
        seat_number VARCHAR(10),
        seat_class VARCHAR(50),
        leg_status VARCHAR(50),
        created_at TIMESTAMP,
        FOREIGN KEY (itinerary_id) REFERENCES Itineraries(itinerary_id),
        FOREIGN KEY (flight_id) REFERENCES FlightSchedule(flight_id)
    );

    -- Passenger Baggage
    CREATE TABLE PassengerBaggage (
        passenger_baggage_id VARCHAR(20) PRIMARY KEY,
        booking_id VARCHAR(20),
        passenger_id VARCHAR(20),
        baggage_type VARCHAR(50),
        bag_weight_kg DECIMAL(5, 2),
        bag_dimensions_cm VARCHAR(50),
        baggage_status VARCHAR(50),
        baggage_tag_number VARCHAR(20),
        created_at TIMESTAMP,
        FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id),
        FOREIGN KEY (passenger_id) REFERENCES Passengers(passenger_id)
    );

    -- Special Baggage
    CREATE TABLE SpecialBaggage (
        special_baggage_id VARCHAR(20) PRIMARY KEY,
        booking_id VARCHAR(20),
        baggage_type VARCHAR(50),
        item_description VARCHAR(500),
        declared_value DECIMAL(10, 2),
        handling_instructions VARCHAR(500),
        created_at TIMESTAMP,
        FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id)
    );

    -- Flight Instances
    CREATE TABLE FlightInstances (
        flight_instance_id VARCHAR(30) PRIMARY KEY,
        flight_id VARCHAR(10),
        flight_date DATE,
        scheduled_departure DATETIME,
        scheduled_arrival DATETIME,
        actual_departure DATETIME,
        actual_arrival DATETIME,
        flight_status VARCHAR(50),
        created_at TIMESTAMP,
        FOREIGN KEY (flight_id) REFERENCES FlightSchedule(flight_id)
    );

    -- Flight Delays
    CREATE TABLE FlightDelays (
        delay_id VARCHAR(20) PRIMARY KEY,
        flight_instance_id VARCHAR(30),
        delay_category VARCHAR(50),
        delay_reason VARCHAR(500),
        delay_minutes INTEGER,
        estimated_departure DATETIME,
        estimated_arrival DATETIME,
        delay_status VARCHAR(50),
        delay_announced_at TIMESTAMP,
        created_at TIMESTAMP,
        FOREIGN KEY (flight_instance_id) REFERENCES FlightInstances(flight_instance_id)
    );

    -- Payments
    CREATE TABLE Payments (
        payment_id VARCHAR(20) PRIMARY KEY,
        booking_id VARCHAR(20),
        payment_amount DECIMAL(10, 2),
        payment_method VARCHAR(50),
        payment_status VARCHAR(50),
        transaction_id VARCHAR(50),
        payment_gateway VARCHAR(50),
        created_at TIMESTAMP,
        FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id)
    );

    -- Refunds
    CREATE TABLE Refunds (
        refund_id VARCHAR(20) PRIMARY KEY,
        booking_id VARCHAR(20),
        refund_reason VARCHAR(50),
        refund_amount DECIMAL(10, 2),
        refund_percentage DECIMAL(5, 2),
        refund_status VARCHAR(50),
        refund_method VARCHAR(50),
        refund_date DATETIME,
        created_at TIMESTAMP,
        FOREIGN KEY (booking_id) REFERENCES Bookings(booking_id)
    );

    -- Connection Rules
    CREATE TABLE ConnectionRules (
        connection_id INTEGER PRIMARY KEY AUTOINCREMENT,
        origin_airport CHAR(3),
        destination_airport CHAR(3),
        min_connection_time_domestic INTEGER,
        min_connection_time_international INTEGER,
        turnaround_time_mins INTEGER,
        created_at TIMESTAMP
    );

    -- Audit Log
    CREATE TABLE AuditLog (
        log_id INTEGER PRIMARY KEY AUTOINCREMENT,
        entity_type VARCHAR(50),
        entity_id VARCHAR(50),
        action VARCHAR(50),
        old_values TEXT,
        new_values TEXT,
        changed_by VARCHAR(100),
        timestamp DATETIME
    );
''')

conn.commit()
print("✅ Database schema created successfully!")

✅ Database schema created successfully!


In [16]:
# ============================================================================
# 3. CREATE MASTER FLIGHT SCHEDULE FROM INDIGO ROUTES
# ============================================================================

fake = Faker('en_IN')

flight_schedule_data = []
flight_id_counter = 1

# Get unique route pairs
unique_routes = {}
for route in indigo_routes:
    key = (route['origin'], route['destination'])
    if key not in unique_routes:
        unique_routes[key] = route

# Create multiple flights per route with different departure times
departure_times = ['06:00', '09:30', '12:00', '15:30', '18:00', '21:00']
aircraft_types = ['A320', 'A321', 'ATR72']
seat_capacities = {'A320': 180, 'A321': 194, 'ATR72': 70}

for (origin, destination), route in unique_routes.items():
    flight_duration_mins = route['duration_mins']
    
    for dep_idx, dep_time in enumerate(departure_times):
        aircraft_type = aircraft_types[dep_idx % len(aircraft_types)]
        seat_capacity = seat_capacities[aircraft_type]
        
        # Calculate arrival time
        dep_hour, dep_min = map(int, dep_time.split(':'))
        arr_hour = (dep_hour + flight_duration_mins // 60) % 24
        arr_min = (dep_min + flight_duration_mins % 60) % 60
        arr_time = f"{arr_hour:02d}:{arr_min:02d}"
        
        flight_id = f"6E{flight_id_counter:04d}"
        
        cursor.execute('''
            INSERT INTO FlightSchedule 
            (flight_id, origin_airport_code, destination_airport_code, 
             departure_time, arrival_time, flight_duration_minutes, 
             aircraft_type, seat_capacity, status, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            flight_id, origin, destination,
            dep_time, arr_time, flight_duration_mins,
            aircraft_type, seat_capacity, 'active', datetime.now()
        ))
        
        flight_id_counter += 1

conn.commit()
print(f"✅ Flight Schedule created with {flight_id_counter - 1} flights")

# Insert Days of Operation (all 7 days for all flights)
flights_df = pd.read_sql_query("SELECT flight_id FROM FlightSchedule", conn)
start_date = '2026-01-01'
end_date = '2046-12-31'

for flight_id in flights_df['flight_id']:
    for day in range(7):  # 0=Sun to 6=Sat
        cursor.execute('''
            INSERT INTO DaysOfOperation 
            (flight_id, day_of_week, effective_from, effective_to)
            VALUES (?, ?, ?, ?)
        ''', (flight_id, day, start_date, end_date))

conn.commit()
print(f"✅ Days of Operation inserted for all flights")

✅ Flight Schedule created with 6762 flights
✅ Days of Operation inserted for all flights


/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/2552362154.py:37: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''


In [17]:
# ============================================================================
# 4. GENERATE SYNTHETIC CUSTOMERS (10,000 customers)
# ============================================================================

print("Generating 10,000 synthetic customers...")

num_customers = 10000
customer_ids = []

for i in range(num_customers):
    customer_id = f"CUST{i+1:08d}"
    customer_ids.append(customer_id)
    
    cursor.execute('''
        INSERT INTO Customers 
        (customer_id, first_name, last_name, email, phone_number, 
         date_of_birth, gender, country_code, city, loyalty_program_id, 
         customer_segment, created_at, updated_at, is_synthetic)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        customer_id,
        fake.first_name(),
        fake.last_name(),
        f"customer{i+1}@indigo.com",  # Ensure unique emails
        fake.phone_number()[:20],
        fake.date_of_birth(minimum_age=18, maximum_age=75),
        random.choice(['M', 'F', 'Other']),
        'IN',
        random.choice(['Delhi', 'Mumbai', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Ahmedabad']),
        f"6EREWARD{random.randint(100000, 999999)}",
        random.choice(['economy', 'premium', 'vip']),
        datetime.now(),
        datetime.now(),
        True
    ))
    
    if (i + 1) % 1000 == 0:
        conn.commit()
        print(f"  Progress: {i + 1}/{num_customers} customers")

conn.commit()
print(f"✅ {num_customers} synthetic customers inserted")

Generating 10,000 synthetic customers...
  Progress: 1000/10000 customers
  Progress: 2000/10000 customers
  Progress: 3000/10000 customers
  Progress: 4000/10000 customers
  Progress: 5000/10000 customers
  Progress: 6000/10000 customers


/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/918540122.py:14: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''
/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/918540122.py:14: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''


  Progress: 7000/10000 customers
  Progress: 8000/10000 customers
  Progress: 9000/10000 customers
  Progress: 10000/10000 customers
✅ 10000 synthetic customers inserted


In [18]:
# ============================================================================
# 5. GENERATE SYNTHETIC BOOKINGS, PNRS, PASSENGERS & BAGGAGE (50,000 bookings)
# ============================================================================

print("Generating 50,000 synthetic bookings with PNRs, passengers, and baggage...")

# Get all flights
flights_df = pd.read_sql_query("SELECT flight_id FROM FlightSchedule", conn)
flights_list = flights_df['flight_id'].tolist()

num_bookings = 50000

for i in range(num_bookings):
    booking_id = f"BK{i+1:08d}"
    
    # Random customer
    customer_id = random.choice(customer_ids)
    
    # Random flight & date (within 20 years from 2026-2046)
    flight_id = random.choice(flights_list)
    random_days_offset = random.randint(0, 365 * 20)
    flight_date = (datetime(2026, 1, 1) + timedelta(days=random_days_offset)).date()
    
    # Number of passengers
    num_passengers = random.randint(1, 4)
    
    # Pricing
    base_fare = random.randint(2000, 15000)
    total_fare = base_fare * num_passengers
    tax = int(total_fare * 0.05)
    discount = int(total_fare * random.uniform(0, 0.15))
    final_amount = total_fare + tax - discount
    
    # PNR code (ensure uniqueness)
    pnr_code = f"{random.choice('ABCDEFGHIJKLMNOPQRSTUVWXYZ')}{i+1:06d}"
    booking_status = random.choice(['confirmed', 'pending_payment', 'cancelled'])
    pnr_status = 'issued' if booking_status == 'confirmed' else 'pending'
    
    # Insert PNR
    cursor.execute('''
        INSERT INTO PNRs 
        (pnr_code, customer_id, pnr_status, issue_date, valid_until, created_at, updated_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (
        pnr_code, customer_id, pnr_status,
        datetime.now(),
        (datetime.now() + timedelta(days=365)).date(),
        datetime.now(), datetime.now()
    ))
    
    # Insert Booking
    cursor.execute('''
        INSERT INTO Bookings 
        (booking_id, pnr_code, customer_id, flight_id, flight_date, 
         total_passengers, booking_status, booking_type, total_fare, 
         tax_charges, discount, final_amount, created_at, updated_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        booking_id, pnr_code, customer_id, flight_id, flight_date,
        num_passengers, booking_status, 'oneway', total_fare,
        tax, discount, final_amount, datetime.now(), datetime.now()
    ))
    
    # Insert Itinerary
    itinerary_id = f"ITN{booking_id[2:]}"
    cursor.execute('''
        INSERT INTO Itineraries 
        (itinerary_id, booking_id, total_legs, journey_type, total_duration_minutes, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (itinerary_id, booking_id, 1, 'direct', 200, datetime.now()))
    
    # Insert Passengers & Baggage for each passenger
    for p in range(num_passengers):
        passenger_id = f"PASS{booking_id[2:]}{p+1:02d}"
        
        cursor.execute('''
            INSERT INTO Passengers 
            (passenger_id, booking_id, first_name, last_name, date_of_birth, 
             gender, passport_number, nationality, passenger_type, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            passenger_id, booking_id,
            fake.first_name(), fake.last_name(),
            fake.date_of_birth(minimum_age=1, maximum_age=75),
            random.choice(['M', 'F', 'Other']),
            f"XXX{random.randint(1000, 9999)}",
            'IN',
            'adult' if p == 0 else random.choice(['adult', 'child']),
            datetime.now()
        ))
        
        # Insert Baggage
        passenger_baggage_id = f"BAG{passenger_id}"
        cursor.execute('''
            INSERT INTO PassengerBaggage 
            (passenger_baggage_id, booking_id, passenger_id, baggage_type, 
             bag_weight_kg, baggage_status, baggage_tag_number, created_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            passenger_baggage_id, booking_id, passenger_id,
            'checked',
            round(random.uniform(10, 25), 1),
            random.choice(['booked', 'checked_in', 'loaded', 'delivered']),
            f"6E{random.randint(1000000, 9999999)}",
            datetime.now()
        ))
    
    # Insert Itinerary Leg
    leg_id = f"LEG{booking_id[2:]}01"
    cursor.execute('''
        INSERT INTO ItineraryLegs 
        (leg_id, itinerary_id, leg_number, flight_id, flight_date, 
         origin_airport, destination_airport, departure_time, arrival_time, 
         seat_number, seat_class, leg_status, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        leg_id, itinerary_id, 1, flight_id, flight_date,
        'DEL', 'BOM',  # Sample route
        '06:00', '08:30',
        f"{random.randint(1, 20)}{random.choice(['A', 'B', 'C', 'D', 'E', 'F'])}",
        random.choice(['economy', 'business']),
        'confirmed',
        datetime.now()
    ))
    
    # Insert Payment
    payment_id = f"PAY{booking_id[2:]}"
    cursor.execute('''
        INSERT INTO Payments 
        (payment_id, booking_id, payment_amount, payment_method, 
         payment_status, transaction_id, payment_gateway, created_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''', (
        payment_id, booking_id, final_amount,
        random.choice(['credit_card', 'debit_card', 'upi', 'netbanking']),
        'success' if booking_status == 'confirmed' else 'pending',
        f"TXN{random.randint(1000000000, 9999999999)}",
        random.choice(['Razorpay', 'PayU', 'CCAvenue']),
        datetime.now()
    ))
    
    if (i + 1) % 5000 == 0:
        conn.commit()
        print(f"  Progress: {i + 1}/{num_bookings} bookings")

conn.commit()
print(f"✅ {num_bookings} bookings with passengers & baggage inserted")

Generating 50,000 synthetic bookings with PNRs, passengers, and baggage...


/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/3064955659.py:40: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''
/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/3064955659.py:40: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''
/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/3064955659.py:52: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''
/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/3064955659.py:52: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''
/var/fol

  Progress: 5000/50000 bookings
  Progress: 10000/50000 bookings
  Progress: 15000/50000 bookings
  Progress: 20000/50000 bookings
  Progress: 25000/50000 bookings
  Progress: 30000/50000 bookings
  Progress: 35000/50000 bookings
  Progress: 40000/50000 bookings
  Progress: 45000/50000 bookings
  Progress: 50000/50000 bookings
✅ 50000 bookings with passengers & baggage inserted


In [19]:
# ============================================================================
# 6. GENERATE FLIGHT INSTANCES & DELAYS (for next 2 years)
# ============================================================================

print("Generating flight instances and delays for 2 years...")

start_generation = datetime(2026, 1, 1)
num_days = 365 * 2  # 2 years of flight instances

delay_counter = 1

for days_ahead in range(0, num_days, 7):  # Generate weekly (optimize for demo)
    current_date = start_generation + timedelta(days=days_ahead)
    
    for flight_id in flights_list[:100]:  # Sample 100 flights to avoid bloat
        flight_instance_id = f"{flight_id}{current_date.strftime('%Y%m%d')}"
        
        # Get flight details
        flight_detail = pd.read_sql_query(
            f"SELECT departure_time, arrival_time FROM FlightSchedule WHERE flight_id = ?",
            conn, params=(flight_id,)
        )
        
        if not flight_detail.empty:
            dep_str = flight_detail.iloc[0]['departure_time']
            arr_str = flight_detail.iloc[0]['arrival_time']
            
            dep_hour, dep_min = map(int, dep_str.split(':'))
            arr_hour, arr_min = map(int, arr_str.split(':'))
            
            scheduled_departure = current_date.replace(hour=dep_hour, minute=dep_min)
            scheduled_arrival = current_date.replace(hour=arr_hour, minute=arr_min)
            
            if arr_hour < dep_hour:  # Next day arrival
                scheduled_arrival += timedelta(days=1)
            
            # Insert Flight Instance
            cursor.execute('''
                INSERT INTO FlightInstances 
                (flight_instance_id, flight_id, flight_date, scheduled_departure, 
                 scheduled_arrival, flight_status, created_at)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (
                flight_instance_id, flight_id, current_date.date(),
                scheduled_departure, scheduled_arrival, 'scheduled', datetime.now()
            ))
            
            # Random delays (5% chance)
            if random.random() < 0.05:
                delay_id = f"DLY{delay_counter:08d}"
                delay_minutes = random.choice([15, 30, 45, 60, 90, 120])
                
                cursor.execute('''
                    INSERT INTO FlightDelays 
                    (delay_id, flight_instance_id, delay_category, delay_reason, 
                     delay_minutes, estimated_departure, estimated_arrival, 
                     delay_status, created_at)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
                ''', (
                    delay_id, flight_instance_id,
                    random.choice(['weather', 'mechanical', 'crew', 'air_traffic']),
                    random.choice(['Heavy rain', 'Aircraft maintenance', 'Crew scheduling', 'Air traffic control']),
                    delay_minutes,
                    scheduled_departure + timedelta(minutes=delay_minutes),
                    scheduled_arrival + timedelta(minutes=delay_minutes),
                    'resolved',
                    datetime.now()
                ))
                
                delay_counter += 1

conn.commit()
print(f"✅ Flight instances and {delay_counter - 1} delays inserted")

Generating flight instances and delays for 2 years...


/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/3522280817.py:38: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''
/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/3522280817.py:38: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''
/var/folders/bg/ql8wn08n167f8j9j723dzcfh0000gn/T/ipykernel_98344/3522280817.py:53: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute('''


✅ Flight instances and 522 delays inserted


In [20]:
# ============================================================================
# 7. DATABASE VERIFICATION & SUMMARY
# ============================================================================

print("\n" + "="*70)
print("DATABASE SUMMARY - INDIGO AIRLINE BOOKING SYSTEM")
print("="*70 + "\n")

# Query table record counts
tables = [
    'Customers', 'FlightSchedule', 'DaysOfOperation', 'PNRs', 'Bookings',
    'Passengers', 'Itineraries', 'ItineraryLegs', 'PassengerBaggage',
    'FlightInstances', 'FlightDelays', 'Payments'
]

total_records = 0
for table in tables:
    count = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table}", conn)
    record_count = count.iloc[0]['count']
    total_records += record_count
    print(f"{table:.<30} {record_count:>10,} records")

print("-" * 70)
print(f"{'TOTAL':.<30} {total_records:>10,} records")
print("\n" + "="*70)

# Sample queries
print("\n📊 SAMPLE DATA QUERIES:\n")

# 1. Top 5 busiest routes
print("1️⃣  TOP 5 BUSIEST ROUTES (by bookings):")
busiest_routes = pd.read_sql_query('''
    SELECT 
        f.origin_airport_code,
        f.destination_airport_code,
        COUNT(b.booking_id) as total_bookings,
        SUM(b.final_amount) as total_revenue
    FROM Bookings b
    JOIN FlightSchedule f ON b.flight_id = f.flight_id
    GROUP BY f.origin_airport_code, f.destination_airport_code
    ORDER BY total_bookings DESC
    LIMIT 5
''', conn)
print(busiest_routes.to_string(index=False))

# 2. Top customers
print("\n\n2️⃣  TOP 5 CUSTOMERS (by spending):")
top_customers = pd.read_sql_query('''
    SELECT 
        c.customer_id,
        c.first_name,
        c.last_name,
        COUNT(b.booking_id) as total_bookings,
        SUM(b.final_amount) as total_spent,
        c.customer_segment
    FROM Customers c
    LEFT JOIN Bookings b ON c.customer_id = b.customer_id
    GROUP BY c.customer_id
    ORDER BY total_spent DESC
    LIMIT 5
''', conn)
print(top_customers.to_string(index=False))

# 3. Delay statistics
print("\n\n3️⃣  FLIGHT DELAY STATISTICS:")
delay_stats = pd.read_sql_query('''
    SELECT 
        delay_category,
        COUNT(*) as delay_count,
        ROUND(AVG(delay_minutes), 1) as avg_delay_mins,
        MAX(delay_minutes) as max_delay_mins
    FROM FlightDelays
    GROUP BY delay_category
    ORDER BY delay_count DESC
''', conn)
print(delay_stats.to_string(index=False))

# 4. Booking status breakdown
print("\n\n4️⃣  BOOKING STATUS BREAKDOWN:")
booking_status = pd.read_sql_query('''
    SELECT 
        booking_status,
        COUNT(*) as count,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM Bookings), 2) as percentage
    FROM Bookings
    GROUP BY booking_status
''', conn)
print(booking_status.to_string(index=False))

# 5. Indigo routes coverage
print("\n\n5️⃣  INDIGO ROUTES COVERAGE:")
routes_coverage = pd.read_sql_query('''
    SELECT 
        COUNT(DISTINCT CONCAT(origin_airport_code, destination_airport_code)) as unique_routes,
        COUNT(DISTINCT origin_airport_code) as origin_airports,
        COUNT(DISTINCT destination_airport_code) as dest_airports,
        COUNT(DISTINCT flight_id) as total_flights
    FROM FlightSchedule
''', conn)
print(routes_coverage.to_string(index=False))

print("\n" + "="*70)
print("\n✅ DATABASE READY FOR OPEN-SOURCE RELEASE!")
print("📁 Database file: indigo_airline.db")
print("📊 All tables populated with synthetic data (Faker library)")
print("🔒 No real customer PII - fully anonymized\n")

# Get database file size
import os
db_size_mb = os.path.getsize('indigo_airline.db') / (1024 * 1024)
print(f"💾 Database size: {db_size_mb:.2f} MB")
print("="*70)


DATABASE SUMMARY - INDIGO AIRLINE BOOKING SYSTEM

Customers.....................     10,000 records
FlightSchedule................      6,762 records
DaysOfOperation...............     47,334 records
PNRs..........................     50,000 records
Bookings......................     50,000 records
Passengers....................    124,836 records
Itineraries...................     50,000 records
ItineraryLegs.................     50,000 records
PassengerBaggage..............    124,836 records
FlightInstances...............     10,500 records
FlightDelays..................        522 records
Payments......................     50,000 records
----------------------------------------------------------------------
TOTAL.........................    574,790 records


📊 SAMPLE DATA QUERIES:

1️⃣  TOP 5 BUSIEST ROUTES (by bookings):
origin_airport_code destination_airport_code  total_bookings  total_revenue
                COK                      DXB              64        1422970
         

In [21]:
# ============================================================================
# 8. USEFUL QUERIES FOR BOOKING SYSTEM
# ============================================================================

print("\n" + "="*70)
print("BOOKING SYSTEM QUERIES - EXAMPLES FOR FRONTEND INTEGRATION")
print("="*70 + "\n")

# Query 1: Search direct flights
print("📌 QUERY 1: Search Direct Flights (Origin → Destination)")
print("-" * 70)
search_origin = 'DEL'
search_destination = 'BOM'
search_date = '2026-02-15'

direct_flights = pd.read_sql_query(f'''
    SELECT 
        f.flight_id,
        f.origin_airport_code,
        f.destination_airport_code,
        f.departure_time,
        f.arrival_time,
        f.flight_duration_minutes,
        f.aircraft_type,
        f.seat_capacity,
        (f.seat_capacity - COALESCE(COUNT(b.booking_id), 0)) as available_seats
    FROM FlightSchedule f
    LEFT JOIN Bookings b ON f.flight_id = b.flight_id AND b.flight_date = ?
    WHERE f.origin_airport_code = ? AND f.destination_airport_code = ?
    GROUP BY f.flight_id
    ORDER BY f.departure_time
''', conn, params=(search_date, search_origin, search_destination))

print(f"Query: Flights from {search_origin} to {search_destination} on {search_date}")
print(direct_flights.to_string(index=False))

# Query 2: Get booking details with customer & passenger info
print("\n\n📌 QUERY 2: Get Booking Details (PNR, Customer, Passengers)")
print("-" * 70)

sample_booking = pd.read_sql_query('''
    SELECT 
        b.booking_id,
        b.pnr_code,
        c.first_name,
        c.last_name,
        c.email,
        b.booking_status,
        b.final_amount,
        b.flight_date
    FROM Bookings b
    JOIN Customers c ON b.customer_id = c.customer_id
    LIMIT 1
''', conn)

if not sample_booking.empty:
    sample_booking_id = sample_booking.iloc[0]['booking_id']
    pnr = sample_booking.iloc[0]['pnr_code']
    print(f"\nSample Booking: {sample_booking_id} (PNR: {pnr})")
    print(sample_booking.to_string(index=False))
    
    # Get passengers for this booking
    print(f"\nPassengers in booking {sample_booking_id}:")
    passengers = pd.read_sql_query('''
        SELECT 
            passenger_id,
            first_name,
            last_name,
            passenger_type,
            date_of_birth
        FROM Passengers
        WHERE booking_id = ?
    ''', conn, params=(sample_booking_id,))
    print(passengers.to_string(index=False))
    
    # Get baggage for this booking
    print(f"\nBaggage in booking {sample_booking_id}:")
    baggage = pd.read_sql_query('''
        SELECT 
            passenger_id,
            baggage_type,
            bag_weight_kg,
            baggage_status,
            baggage_tag_number
        FROM PassengerBaggage
        WHERE booking_id = ?
    ''', conn, params=(sample_booking_id,))
    print(baggage.to_string(index=False))

# Query 3: Find connecting flights
print("\n\n📌 QUERY 3: Find Connecting Flights (with connection time validation)")
print("-" * 70)
print("""
-- Example: Find flights from DEL to London (multi-leg with connection)
-- Logic: Find all combinations of DEL→X and X→LHR where X is a hub
-- Minimum connection time: 90min domestic, 120min international
SELECT 
    leg1.flight_id as outbound_flight,
    leg2.flight_id as connecting_flight,
    leg1.arrival_time as arrival_at_hub,
    leg2.departure_time as departure_from_hub,
    CAST(
        (strftime('%s', leg2.departure_time) - strftime('%s', leg1.arrival_time)) / 60 
        AS INTEGER
    ) as connection_time_mins
FROM FlightSchedule leg1
JOIN FlightSchedule leg2 ON leg1.destination_airport_code = leg2.origin_airport_code
WHERE leg1.origin_airport_code = 'DEL'
    AND leg2.destination_airport_code = 'LHR'
    AND (strftime('%s', leg2.departure_time) - strftime('%s', leg1.arrival_time)) / 60 >= 120
ORDER BY connection_time_mins ASC;
""")

# Query 4: Get flights with delays
print("\n📌 QUERY 4: Get Delayed Flights in Last 30 Days")
print("-" * 70)

delayed_flights = pd.read_sql_query('''
    SELECT 
        fi.flight_id,
        fi.flight_date,
        fi.scheduled_departure,
        fd.delay_reason,
        fd.delay_minutes,
        fd.estimated_departure
    FROM FlightDelays fd
    JOIN FlightInstances fi ON fd.flight_instance_id = fi.flight_instance_id
    WHERE fi.flight_date >= date('now', '-30 days')
    ORDER BY fi.flight_date DESC, fi.scheduled_departure
    LIMIT 10
''', conn)

if not delayed_flights.empty:
    print(delayed_flights.to_string(index=False))
else:
    print("No delays in the last 30 days")

print("\n" + "="*70)
print("✅ Notebook Complete! All tables populated and queries ready.")
print("="*70)


BOOKING SYSTEM QUERIES - EXAMPLES FOR FRONTEND INTEGRATION

📌 QUERY 1: Search Direct Flights (Origin → Destination)
----------------------------------------------------------------------
Query: Flights from DEL to BOM on 2026-02-15
flight_id origin_airport_code destination_airport_code departure_time arrival_time  flight_duration_minutes aircraft_type  seat_capacity  available_seats
   6E2263                 DEL                      BOM          06:00        08:15                      135          A320            180              180
   6E2264                 DEL                      BOM          09:30        11:45                      135          A321            194              194
   6E2265                 DEL                      BOM          12:00        14:15                      135         ATR72             70               70
   6E2266                 DEL                      BOM          15:30        17:45                      135          A320            180              1